# 02. Модель Pb и проверка LOWO

Целевая переменная: log(Pb) минус медиана log(Pb) по складу. Модель: Random Forest (300 деревьев, min_samples_leaf=3). Проверка: Leave-One-Warehouse-Out, то есть обучение на 9 складах и проверка на десятом.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.3})
import matplotlib.pyplot as plt
import mltj
sites, pts, cov = mltj.load()
print(len(sites), 'складов,', len(pts), 'точек')

## Результат и разброс по случайному зерну

In [ ]:
y = mltj.target(pts,'Pb')
rows=[]
for name,f in [('базовый набор (положение точки)',mltj.BASE),('+ расстояние до воды',mltj.BASE+['dist_to_water_m']),('+ вода + pH (продакшн)',mltj.FEATURES)]:
    r=[mltj.lowo_r2(pts,y,f,seed=s)[0] for s in range(8)]
    rows.append((name,np.mean(r),np.min(r),np.max(r)))
pd.DataFrame(rows,columns=['набор признаков','R² среднее','мин','макс']).round(3)

Разброс определяется случайным зерном леса. Поэтому в отчётах указывается диапазон (0,16–0,17), а не одно число.

## Предсказание и факт на складе, не участвовавшем в обучении

In [ ]:
pred = mltj.lowo_predict(pts,y,mltj.FEATURES,seed=0)
fig,ax=plt.subplots(figsize=(4.6,4.2)); ax.scatter(y,pred,s=10,alpha=.6,c='#3b6ea5'); lim=[min(y.min(),pred.min()),max(y.max(),pred.max())]; ax.plot(lim,lim,'k--',lw=1); ax.set_xlabel('факт: log(Pb) - медиана склада'); ax.set_ylabel('прогноз LOWO'); plt.show()

## R² по складам

R² на отдельном складе может быть отрицательным; сводный R² считается по всем точкам.

In [ ]:
from sklearn.metrics import r2_score
per=pd.DataFrame({'склад':pts.site_name,'y':y,'p':pred}).groupby('склад').apply(lambda d: pd.Series({'точек':len(d),'R²':r2_score(d.y,d.p) if len(d)>2 else np.nan}))
per.round(3).sort_values('R²')

## Кривая обучения: сколько складов нужно

Модель обучается на k случайных складах из оставшихся девяти и проверяется на десятом. Рост R² с числом складов показывает, что ограничение проекта — объём данных, а не признаки. Экстраполяция за 9 складов — гипотеза.

In [ ]:
import random
from sklearn.ensemble import RandomForestRegressor
F=mltj.FEATURES; sites_l=list(pts.site_id.unique()); res={}
for k in (3,5,7,9):
    vals=[]
    for rep in range(12):
        rnd=random.Random(rep*100+k); pred=np.zeros(len(y))
        for te_s in sites_l:
            tr_s=rnd.sample([s for s in sites_l if s!=te_s],k)
            tr=pts.site_id.isin(tr_s).values; te=(pts.site_id==te_s).values
            pred[te]=RandomForestRegressor(n_estimators=100,min_samples_leaf=3,random_state=rep).fit(pts.loc[tr,F].values,y[tr]).predict(pts.loc[te,F].values)
        vals.append(r2_score(y,pred))
    res[k]=(np.mean(vals),np.std(vals))
fig,ax=plt.subplots(figsize=(5,3.2)); ks=list(res); ax.errorbar(ks,[res[k][0] for k in ks],[res[k][1] for k in ks],marker='o'); ax.set_xlabel('складов в обучении'); ax.set_ylabel('LOWO R²'); plt.show()
pd.DataFrame(res,index=['R²','sd']).round(3)